In [267]:
import pandas as pd
import folium
from folium.plugins import BeautifyIcon
import webbrowser
import os


In [268]:
nodes_final = pd.read_excel("danish_grid_graph_ready.xlsx", sheet_name="nodes")
edges_final = pd.read_excel("danish_grid_graph_ready.xlsx", sheet_name="edges")

In [269]:
m = folium.Map(location=[56.2, 10.0], zoom_start=7, tiles="CartoDB Positron")

In [270]:
def get_node_style(row):
    name = str(row["name"]).lower()
    voltage = row["voltage_kv"]

    if "hvdc" in name:
        return "blue", "circle"
    if "windoff" in name or "havmølle" in name:
        return "yellow", "circle"
    if "værket" in name:
        return "black", "star"

    if pd.notna(voltage):
        if voltage >= 300:
            return "red", "circle"
        elif voltage >= 200:
            return "green", "circle"

    return "gray", "circle"

In [271]:
for _, row in nodes_final.iterrows():
    if pd.notna(row["lat"]) and pd.notna(row["lon"]):

        color, shape = get_node_style(row)

        popup_text = f"""
        <b>{row['name']}</b><br>
        Voltage: {row['voltage_kv']} kV<br>
        Supply: {row['supply']:.1f} MW<br>
        Demand: {row['demand']:.1f} MW<br>
        Source: {row['source']}
        """

        if shape == "star":
            folium.Marker(
                location=[row["lat"], row["lon"]],
                icon=folium.Icon(color="black", icon="star"),
                popup=popup_text
            ).add_to(m)
        else:
            folium.CircleMarker(
                location=[row["lat"], row["lon"]],
                radius=4,
                color=color,
                fill=True,
                fill_opacity=0.8,
                popup=popup_text
            ).add_to(m)

In [272]:
def get_edge_style(row):
    voltage = row.get("voltage_kv", None)
    line_type = str(row.get("line_type", "")).lower()
    edge_type = row.get("edge_type", "")

    if edge_type == "hvdc":
        return "blue", "5,5"

    is_cable = "cable" in line_type

    if pd.notna(voltage):
        if voltage >= 300:
            return ("red", "5,5") if is_cable else ("red", None)
        elif voltage >= 200:
            return ("green", "5,5") if is_cable else ("green", None)
        else:
            return ("black", "5,5") if is_cable else ("black", None)

    return "gray", None


for _, row in edges_final.iterrows():
    n1 = nodes_final[nodes_final["bus_index"] == row["node1"]]
    n2 = nodes_final[nodes_final["bus_index"] == row["node2"]]

    if len(n1) > 0 and len(n2) > 0:
        latlon1 = n1.iloc[0][["lat", "lon"]]
        latlon2 = n2.iloc[0][["lat", "lon"]]

        if pd.notna(latlon1["lat"]) and pd.notna(latlon2["lat"]):

            color, dash = get_edge_style(row)

            folium.PolyLine(
                locations=[
                    [latlon1["lat"], latlon1["lon"]],
                    [latlon2["lat"], latlon2["lon"]]
                ],
                color=color,
                weight=2,
                opacity=0.7,
                dash_array=dash
            ).add_to(m)

In [273]:
m.save("danish_grid_map.html")
webbrowser.open("file://" + os.path.realpath("danish_grid_map.html"))

True